# Laboratorium

## Funkcje agregujące w SQL

**Temat zajęć:** obliczenia zbiorcze, grupowanie danych oraz filtrowanie grup za pomocą `GROUP BY` i `HAVING`.


## Cele zajęć

Po zakończeniu zajęć student powinien umieć:

1. wyjaśnić, czym są funkcje agregujące i kiedy warto ich używać,
2. stosować podstawowe agregaty: `COUNT`, `SUM`, `AVG`, `MIN`, `MAX`,
3. poprawnie łączyć funkcje agregujące z `GROUP BY`,
4. odróżniać filtrowanie wierszy za pomocą `WHERE` od filtrowania grup za pomocą `HAVING`,
5. interpretować wyniki zapytań agregujących i nadawać im czytelne aliasy,
6. unikać typowych błędów, np. wyboru kolumny spoza `GROUP BY` bez agregacji.


## 1. Czym są funkcje agregujące?

Funkcje agregujące wykonują obliczenia na wielu wierszach i zwracają jedną wartość dla całego zbioru lub jedną wartość dla każdej grupy.

Najczęściej używane funkcje agregujące w SQL:

| Funkcja | Znaczenie | Przykład użycia |
|---|---|---|
| `COUNT(*)` | liczba wszystkich wierszy | liczba klientów |
| `COUNT(kolumna)` | liczba niepustych wartości w kolumnie | liczba klientów z podanym adresem e-mail |
| `COUNT(DISTINCT kolumna)` | liczba unikalnych wartości | liczba różnych krajów |
| `SUM(kolumna)` | suma wartości liczbowych | suma płatności |
| `AVG(kolumna)` | średnia wartość | średnia długość filmu |
| `MIN(kolumna)` | najmniejsza wartość | najkrótszy film |
| `MAX(kolumna)` | największa wartość | najdłuższy film |

Przykład:

```sql
SELECT 
    COUNT(*) AS liczba_filmow,
    MIN(length) AS najkrotszy_film,
    MAX(length) AS najdluzszy_film,
    ROUND(AVG(length), 2) AS srednia_dlugosc
FROM film;
```


## 2. Ważna zasada: wartości `NULL`

Większość funkcji agregujących pomija wartości `NULL`.

Szczególnie ważna jest różnica między:

```sql
COUNT(*)
```

a

```sql
COUNT(nazwa_kolumny)
```

`COUNT(*)` liczy wszystkie wiersze, natomiast `COUNT(nazwa_kolumny)` liczy tylko te wiersze, w których dana kolumna nie ma wartości `NULL`.

Przykład:

```sql
SELECT
    COUNT(*) AS liczba_wierszy,
    COUNT(email) AS liczba_klientow_z_emailem
FROM customer;
```


## 3. Grupowanie danych: `GROUP BY`

Sama funkcja agregująca oblicza jedną wartość dla całego wyniku zapytania. Jeżeli chcemy policzyć osobną wartość dla każdej grupy, używamy `GROUP BY`.

Schemat:

```sql
SELECT
    kolumna_grupujaca,
    funkcja_agregujaca(kolumna) AS alias
FROM tabela
GROUP BY kolumna_grupujaca;
```

Przykład: liczba filmów w każdej kategorii.

```sql
SELECT
    c.name AS kategoria,
    COUNT(*) AS liczba_filmow
FROM category c
JOIN film_category fc ON fc.category_id = c.category_id
JOIN film f ON f.film_id = fc.film_id
GROUP BY c.name
ORDER BY liczba_filmow DESC;
```

Zasada: każda kolumna w `SELECT`, która nie jest użyta wewnątrz funkcji agregującej, powinna znaleźć się w `GROUP BY`.


## 4. `WHERE` a `HAVING`

`WHERE` filtruje pojedyncze wiersze **przed grupowaniem**.

`HAVING` filtruje całe grupy **po grupowaniu**.

Przykład: kraje, w których jest więcej niż 20 klientów.

```sql
SELECT
    co.country AS kraj,
    COUNT(*) AS liczba_klientow
FROM customer cu
JOIN address a ON a.address_id = cu.address_id
JOIN city ci ON ci.city_id = a.city_id
JOIN country co ON co.country_id = ci.country_id
GROUP BY co.country
HAVING COUNT(*) > 20
ORDER BY liczba_klientow DESC;
```

Przykład z `WHERE` i `HAVING` razem:

```sql
SELECT
    co.country AS kraj,
    COUNT(*) AS liczba_aktywnych_klientow
FROM customer cu
JOIN address a ON a.address_id = cu.address_id
JOIN city ci ON ci.city_id = a.city_id
JOIN country co ON co.country_id = ci.country_id
WHERE cu.active = 1
GROUP BY co.country
HAVING COUNT(*) > 10
ORDER BY liczba_aktywnych_klientow DESC;
```

W tym przykładzie:

- `WHERE cu.active = 1` wybiera tylko aktywnych klientów,
- `GROUP BY co.country` grupuje ich według kraju,
- `HAVING COUNT(*) > 10` zostawia tylko kraje z więcej niż 10 aktywnymi klientami.


## 5. Kolejność logicznego wykonywania zapytania

Warto pamiętać, że SQL nie jest wykonywany dokładnie w takiej kolejności, w jakiej zapisujemy klauzule.

Uproszczona kolejność logiczna:

1. `FROM` oraz `JOIN`
2. `WHERE`
3. `GROUP BY`
4. funkcje agregujące
5. `HAVING`
6. `SELECT`
7. `ORDER BY`
8. `LIMIT`

Dlatego nie można używać funkcji agregującej w `WHERE`:

```sql
-- BŁĘDNIE
SELECT store_id, COUNT(*) AS liczba_klientow
FROM customer
WHERE COUNT(*) > 100
GROUP BY store_id;
```

Poprawnie:

```sql
SELECT store_id, COUNT(*) AS liczba_klientow
FROM customer
GROUP BY store_id
HAVING COUNT(*) > 100;
```


## 6. `DISTINCT` w funkcjach agregujących

`DISTINCT` pozwala policzyć tylko unikalne wartości.

Przykład: liczba różnych miast, w których mieszkają klienci.

```sql
SELECT
    COUNT(DISTINCT city_id) AS liczba_miast
FROM address;
```

Przykład: liczba unikalnych klientów, którzy dokonali płatności.

```sql
SELECT
    COUNT(DISTINCT customer_id) AS liczba_placacych_klientow
FROM payment;
```

Bez `DISTINCT` wynik mógłby być zawyżony, ponieważ jeden klient może mieć wiele płatności.


## 7. Dodatkowe funkcje agregujące w PostgreSQL

Oprócz standardowych agregatów PostgreSQL udostępnia także funkcje przydatne w analizie danych, między innymi:

| Funkcja | Zastosowanie |
|---|---|
| `STRING_AGG(kolumna, separator)` | łączy wartości tekstowe w jeden napis |
| `ARRAY_AGG(kolumna)` | tworzy tablicę z wartości |
| `BOOL_AND(kolumna_logiczna)` | sprawdza, czy wszystkie wartości są prawdziwe |
| `BOOL_OR(kolumna_logiczna)` | sprawdza, czy przynajmniej jedna wartość jest prawdziwa |

Przykład: lista przykładowych filmów w każdej kategorii.

```sql
SELECT
    c.name AS kategoria,
    STRING_AGG(f.title, ', ' ORDER BY f.title) AS filmy
FROM category c
JOIN film_category fc ON fc.category_id = c.category_id
JOIN film f ON f.film_id = fc.film_id
GROUP BY c.name
ORDER BY c.name;
```

Uwaga: przy dużej liczbie rekordów taki wynik może być bardzo długi, dlatego w raportach często ogranicza się liczbę prezentowanych elementów.


## 8. Typowe błędy i jak ich unikać

### Błąd 1: kolumna w `SELECT` nie jest ani agregowana, ani grupowana

```sql
-- BŁĘDNIE
SELECT category_id, title, COUNT(*)
FROM film
GROUP BY category_id;
```

Jeżeli używasz `GROUP BY`, każda zwykła kolumna w `SELECT` musi być częścią grupowania albo musi zostać użyta w funkcji agregującej.

### Błąd 2: użycie `WHERE` zamiast `HAVING`

```sql
-- BŁĘDNIE
SELECT customer_id, SUM(amount) AS suma_platnosci
FROM payment
WHERE SUM(amount) > 100
GROUP BY customer_id;
```

Poprawnie:

```sql
SELECT customer_id, SUM(amount) AS suma_platnosci
FROM payment
GROUP BY customer_id
HAVING SUM(amount) > 100;
```

### Błąd 3: brak aliasów

Czytelniejszy wynik:

```sql
SELECT
    customer_id,
    ROUND(SUM(amount), 2) AS laczna_kwota
FROM payment
GROUP BY customer_id
ORDER BY laczna_kwota DESC;
```


## 9. Przykłady do omówienia na zajęciach

### Przykład 1. Średni koszt wypożyczenia / płatności

```sql
SELECT
    ROUND(AVG(amount), 2) AS srednia_kwota
FROM payment;
```

### Przykład 2. Długości filmów, które występują więcej niż raz

```sql
SELECT
    length AS dlugosc_filmu,
    COUNT(*) AS liczba_filmow
FROM film
GROUP BY length
HAVING COUNT(*) > 1
ORDER BY liczba_filmow DESC, dlugosc_filmu;
```

### Przykład 3. Liczba klientów w miastach

```sql
SELECT
    ci.city AS miasto,
    COUNT(*) AS liczba_klientow
FROM customer cu
JOIN address a ON a.address_id = cu.address_id
JOIN city ci ON ci.city_id = a.city_id
GROUP BY ci.city
ORDER BY liczba_klientow DESC, miasto;
```

### Przykład 4. Sklepy mające więcej niż 100 i mniej niż 300 klientów

```sql
SELECT
    store_id,
    COUNT(*) AS liczba_klientow
FROM customer
GROUP BY store_id
HAVING COUNT(*) > 100 AND COUNT(*) < 300
ORDER BY liczba_klientow DESC;
```

### Przykład 5. Klienci, którzy obejrzeli filmy trwające łącznie ponad 200 godzin

W bazie DVD Rental długość filmu zwykle jest zapisana w minutach, dlatego wynik dzielimy przez `60.0`.

```sql
SELECT
    cu.customer_id,
    cu.first_name,
    cu.last_name,
    ROUND(SUM(f.length) / 60.0, 2) AS liczba_godzin
FROM customer cu
JOIN rental r ON r.customer_id = cu.customer_id
JOIN inventory i ON i.inventory_id = r.inventory_id
JOIN film f ON f.film_id = i.film_id
GROUP BY cu.customer_id, cu.first_name, cu.last_name
HAVING SUM(f.length) / 60.0 > 200
ORDER BY liczba_godzin DESC;
```


## 10. Zadania wprowadzające

Wykonaj zapytania w DBMS. Dla każdego zadania nadaj kolumnom czytelne aliasy i posortuj wynik tak, aby był łatwy do interpretacji.

1. Oblicz średni koszt pojedynczej płatności / wypożyczenia.
2. Znajdź długości filmów, które występują w bazie więcej niż raz. Wyświetl długość filmu oraz liczbę filmów o tej długości.
3. Znajdź miasta, w których mieszka więcej niż jeden klient.
4. Oblicz i wyświetl liczbę filmów we wszystkich kategoriach.
5. Wyświetl liczbę klientów pogrupowanych według kraju.
6. Wyświetl identyfikatory sklepów, które mają więcej niż 100 i mniej niż 300 klientów.
7. Wybierz klientów, którzy obejrzeli filmy trwające łącznie ponad 200 godzin.
8. Oblicz łączną i średnią wartość płatności dla każdego klienta.
9. Oblicz średnią długość filmu w każdej kategorii.
10. Znajdź długość najdłuższego tytułu filmowego w każdej kategorii.
11. Znajdź najdłuższy film w każdej kategorii. Porównaj wynik z zadaniem 10 i wyjaśnij różnicę między długością tytułu a długością filmu.


In [1]:
import pandas as pd
from sqlalchemy import URL, create_engine, text

# TODO: Uzupełnij dane do połączenia z bazą danych
db_url = URL.create(
    drivername="postgresql+psycopg2",
    username="postgres",
    password="1234",
    host="localhost",
    port=5432,
    database="dvdrental",
)

try:
    engine = create_engine(db_url)
    connection = engine.connect()
    print("Pomyślnie nawiązano połączenie z bazą danych.")
except Exception as e:
    print(f"Błąd podczas łączenia z bazą danych: {e}")

Pomyślnie nawiązano połączenie z bazą danych.


In [2]:
sql_query = text("""--sql
    SELECT ROUND(AVG(amount), 2) AS średni_koszt
    FROM payment
""")

df = pd.read_sql(sql_query, con=engine)
display(df)

,średni_koszt
0,4.2


In [3]:
sql_query = text("""--sql
    SELECT length, COUNT(*) as liczba_filmow
    FROM film
    GROUP BY length
    ORDER BY length
""")

df = pd.read_sql(sql_query, con=engine)
display(df)

,length,liczba_filmow
0,46,5
1,47,7
2,48,11
3,49,5
4,50,9
...,...,...
135,181,10
136,182,6
137,183,5
138,184,8


In [4]:
sql_query = text("""--sql
    SELECT ci.city AS miasto, COUNT(*) AS liczba_klientow
    FROM customer cu
    JOIN address a ON a.address_id = cu.address_id
    JOIN city ci ON ci.city_id = a.city_id
    GROUP BY ci.city
    HAVING COUNT(*) > 1
    ORDER BY liczba_klientow DESC, miasto;
""")

df = pd.read_sql(sql_query, con=engine)
display(df)

,miasto,liczba_klientow
0,Aurora,2
1,London,2


In [5]:
sql_query = text("""--sql
    SELECT cat.name, COUNT(*) AS liczba_klientów
    FROM category cat
    JOIN film_category fcat ON fcat.category_id = cat.category_id
    JOIN film f ON f.film_id = fcat.film_id
    GROUP BY cat.name
""")

df = pd.read_sql(sql_query, con=engine)
display(df)

,name,liczba_klientów
0,Family,69
1,Games,61
2,Animation,66
3,Classics,57
4,Documentary,68
5,New,63
6,Sports,74
7,Children,60
8,Music,51
9,Travel,57


In [6]:
sql_query = text("""--sql
    SELECT co.country AS państwo, COUNT(*) AS liczba_klientów
    FROM customer cu
    JOIN address a ON a.address_id = cu.address_id
    JOIN city ci ON ci.city_id = a.city_id
    JOIN country co ON co.country_id = ci.country_id
    GROUP BY co.country
""")

df = pd.read_sql(sql_query, con=engine)
display(df)

,państwo,liczba_klientów
0,Thailand,3
1,"Virgin Islands, U.S.",1
2,Faroe Islands,1
3,Bangladesh,3
4,Indonesia,14
...,...,...
103,Tanzania,3
104,Poland,8
105,Taiwan,10
106,Greenland,1


In [7]:
sql_query = text("""--sql
    SELECT 
        c.store_id, 
        COUNT(c.customer_id) AS liczba_klientow
    FROM customer c
    GROUP BY c.store_id
    HAVING COUNT(c.customer_id) > 100 AND COUNT(c.customer_id) < 300;
""")

df = pd.read_sql(sql_query, con=engine)
display(df)

,store_id,liczba_klientow
0,2,273


## 12. Checklista poprawnego rozwiązania

Przed oddaniem ćwiczenia sprawdź:

- czy zapytanie uruchamia się bez błędów,
- czy wszystkie kolumny w `SELECT` są poprawnie agregowane albo znajdują się w `GROUP BY`,
- czy filtrowanie pojedynczych wierszy jest w `WHERE`,
- czy filtrowanie grup jest w `HAVING`,
- czy wyniki mają czytelne aliasy,
- czy sortowanie pomaga zinterpretować wynik,
- czy potrafisz krótko wyjaśnić, co oznacza wynik zapytania.


## 14. Zadanie implementacyjne

Zaimplementuj funkcje w pliku `main.py` zgodnie z listą zadań.

In [9]:
import main
from main import number_films_in_category, number_film_by_length, avg_amount_by_length, client_by_sum_length, category_statistic_length

df = (number_films_in_category(1))
df

Pomyślnie nawiązano połączenie z bazą danych.


,category,count
0,Action,64


In [11]:
df = (number_film_by_length(45, 50))
df

,length,count
0,46,5
1,47,7
2,48,11
3,49,5
4,50,9


In [12]:
df = (avg_amount_by_length(48))
df

,length,avg
0,48,4.295389


In [13]:
df = (client_by_sum_length(1200))
df

,first_name,last_name,sum
0,Brian,Wyman,1265
1,Antonio,Meek,1451
2,Leona,Obrien,1588
3,Katherine,Rivera,1615
4,Tiffany,Jordan,1667
...,...,...,...
594,Clara,Shaw,4808
595,Wesley,Bull,4808
596,Tammy,Sanders,5065
597,Eleanor,Hunt,5360


In [14]:
df = (category_statistic_length("Action"))
df

,category,avg,sum,min,max
0,Action,111.61,7143,47,185
